In [1]:
import pandas as pd
import numpy as np

In [2]:
def get_closest_row(df, target_time):
    df = df.copy()

    df["time_diff"] = (
        df["tracked_at"] - target_time
    ).abs()

    return df.sort_values("time_diff").iloc[0]

In [3]:
def build_dataset():

    # ===== LOAD COHORT =====
    cohort = pd.read_csv("cohort4.csv")

    cohort["cohort_id"] = "cohort4"

    # ===== LOAD TRACKING =====
    tracking = pd.read_csv("tracking_log4.csv")

    # ===== CONVERT DATETIME =====
    cohort["published_at"] = pd.to_datetime(
        cohort["published_at"],
        utc=True
    )

    tracking["tracked_at"] = pd.to_datetime(
        tracking["tracked_at"], 
        utc=True
    )

    final_rows = []

    # ===== BUILD FEATURES =====
    for vid in cohort["video_id"].unique():

        vid_track = tracking[
            tracking["video_id"] == vid
        ]

        if vid_track.empty:
            continue

        try:

            row_cohort = cohort[
                cohort["video_id"] == vid
            ].iloc[0]

            start_time = row_cohort["published_at"]

            target_6h = start_time + pd.Timedelta(hours=6)
            target_24h = start_time + pd.Timedelta(hours=24)

            row_6h = get_closest_row(
                vid_track,
                target_6h
            )

            row_24h = get_closest_row(
                vid_track,
                target_24h
            )

            final_rows.append({

                "video_id": vid,
                "cohort_id": row_cohort["cohort_id"],

                "views_6h": int(row_6h["views"]),
                "likes_6h": int(row_6h["likes"]),
                "comments_6h": int(row_6h["comments"]),

                "views_24h": int(row_24h["views"]),
                "likes_24h": int(row_24h["likes"]),
                "comments_24h": int(row_24h["comments"]),
            })

        except Exception as e:
            print(f"Skip {vid} karena error: {e}")
            continue

    # ===== FINAL DATAFRAME =====
    df_final = pd.DataFrame(final_rows)

    # ===== DATA CLEANING ======
    df_final = df_final.drop_duplicates(
        subset=["video_id"]
    )
    # hapus view 0
    df_final = df_final[df_final['views_6h'] > 0]

    # hapus anomali sperti views tinggi like 0
    df_final = df_final[~((df_final["likes_6h"] == 0) & (df_final["views_6h"] > 10000))]

    # ===== NORMALIZATION =====
    df_final["views_log"] = np.log1p(df_final["views_24h"])

    df_final["likes_log"] = np.log1p(df_final["likes_24h"])

    df_final["comments_log"] = np.log1p(df_final["comments_24h"])

    # ===== VIRAL SCORE =====
    df_final["viral_score"] = (
        0.6 * df_final["views_log"] +
        0.3 * df_final["likes_log"] +
        0.1 * df_final["comments_log"]
    )

    # ===== VIRAL LABEL =====
    threshold = df_final[
        "viral_score"
    ].quantile(0.8)

    df_final["viral"] = (
        df_final["viral_score"] >= threshold
    ).astype(int)

    print(f"Viral threshold: {threshold}")

    print(
        df_final["viral"]
        .value_counts()
    )

    # ===== SAVE =====
    df_final.to_csv(
        "dataset4.csv",
        index=False
    )

    print(
        f"✅ Dataset berhasil dibuat: {len(df_final)} video"
    )

In [4]:
if __name__ == "__main__":
    build_dataset()

Viral threshold: 8.721657510301144
viral
0    120
1     30
Name: count, dtype: int64
✅ Dataset berhasil dibuat: 150 video
